# Loan Prediction: Commented Jupyter Notebook

This notebook demonstrates a complete loan-prediction workflow using `loanP_train.csv`. Each section explains what the code does and why the step is necessary.

## 1. Import the required libraries

Pandas is used to load and clean the dataset. Scikit-learn provides tools for splitting the data, scaling numeric features, training logistic regression, and measuring model performance.

In [ ]:
# Import Path so the notebook can use a clear, reusable CSV file path.
from pathlib import Path

# Import pandas for reading, inspecting, cleaning, and transforming tabular data.
import pandas as pd

# Import machine-learning utilities used later in the notebook.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Load the CSV dataset

The CSV file is read into a pandas DataFrame named `df`. A DataFrame makes it easier to inspect columns, locate missing values, and prepare the data for modeling.

In [ ]:
# Define the path of the CSV file stored beside this notebook.
csv_path = Path("loanP_train.csv")

# Load the loan application records into a DataFrame.
df = pd.read_csv(csv_path)

# Display the first five records to confirm that the file loaded correctly.
df.head()

## 3. Understand the dataset

Before cleaning data, it is important to understand its size, column types, missing-value counts, and target distribution. These checks determine which preprocessing methods are appropriate.

In [ ]:
# Show the number of rows and columns in the original dataset.
print("Dataset shape:", df.shape)

# Display each column's data type to separate numeric and categorical fields.
print("\nColumn data types:")
print(df.dtypes)

# Count missing values so that no incomplete column is overlooked.
print("\nMissing values by column:")
print(df.isna().sum())

# Review the balance of approved and declined applications.
print("\nLoan status distribution:")
print(df["Loan_Status"].value_counts())

## 4. Check and remove duplicate records

Duplicate rows can give repeated applications too much influence. This section counts duplicates, removes them if any exist, and resets the row index.

In [ ]:
# Count complete duplicate rows before removing them.
duplicate_count = int(df.duplicated().sum())
print("Duplicate rows found:", duplicate_count)

# Keep only one copy of every identical row and rebuild a clean index.
df = df.drop_duplicates().reset_index(drop=True)

## 5. Fill missing values

Categorical fields are filled with their most frequent value (mode). Numeric fields are filled with their median, which is less sensitive to unusually large income or loan values than the mean.

In [ ]:
# Identify text/category columns and numeric columns automatically.
categorical_columns = df.select_dtypes(include="object").columns
numeric_columns = df.select_dtypes(include="number").columns

# Fill each categorical column with its most common non-missing value.
for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

# Fill each numeric column with its median value.
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())

# Confirm that the cleaning step removed every missing value.
print("Remaining missing values:", int(df.isna().sum().sum()))

## 6. Prepare features and the target

`Loan_ID` is an identifier rather than a predictive characteristic, so it is removed from the model inputs. `Dependents` is converted to a numeric field, and the target is mapped from Y/N to 1/0.

In [ ]:
# Convert the '3+' category into 3 and then convert the full column to integers.
df["Dependents"] = df["Dependents"].replace("3+", "3").astype(int)

# Separate the predictor columns from the result the model must learn.
X = df.drop(columns=["Loan_ID", "Loan_Status"])
y = df["Loan_Status"].map({"N": 0, "Y": 1})

# Validate that every target value was converted successfully.
assert y.notna().all(), "Loan_Status contains an unexpected category."

## 7. Encode categorical predictor columns

Machine-learning models require numeric inputs. One-hot encoding creates binary indicator columns for categories such as gender, education, and property area without treating their labels as ordered values.

In [ ]:
# Convert categorical predictors into 0/1 indicator columns.
# drop_first=True removes one reference category from each group.
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# Display the model-ready column names and final feature-table size.
print("Encoded feature shape:", X_encoded.shape)
print("Encoded columns:", X_encoded.columns.tolist())

## 8. Split the data into training and testing sets

The model trains on 80% of the applications and is tested on the remaining 20%. Stratification preserves the approved/declined proportion in both sets, while `random_state=42` makes the result reproducible.

In [ ]:
# Create independent training and testing subsets.
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 9. Standardize continuous numeric features

Income, loan amount, and loan term use different scales. Standardizing continuous numeric columns gives them a mean near 0 and a standard deviation near 1. The scaler is fitted only on training data to prevent test-data leakage.

In [ ]:
# List the continuous/count fields that should be standardized.
columns_to_scale = [
    "Dependents",
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
]

# Copy the split data so the unscaled encoded table remains available.
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Learn scaling values from training data, then apply the same values to test data.
scaler = StandardScaler()
X_train_scaled[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test_scaled[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

## 10. Train the logistic regression model

Logistic regression is appropriate because `Loan_Status` has two possible outcomes. The model learns relationships between applicant characteristics and the probability of approval.

In [ ]:
# Create and train the classification model.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict loan outcomes for applications the model did not train on.
y_pred = model.predict(X_test_scaled)

## 11. Evaluate model performance

Accuracy shows the overall percentage of correct predictions. The confusion matrix separates correct and incorrect approvals/declines, while the classification report provides precision, recall, and F1-score for each outcome.

In [ ]:
# Calculate and display the model's test-set performance.
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Declined", "Approved"]))

## 12. Save the cleaned dataset

The cleaned DataFrame is exported to a new CSV file. This preserves the original source file while making the completed cleaning work reusable outside the notebook.

In [ ]:
# Save all cleaned values without writing the DataFrame index as an extra column.
cleaned_csv_path = Path("loanP_train_cleaned.csv")
df.to_csv(cleaned_csv_path, index=False)

# Perform final checks and show where the cleaned file was saved.
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0
print("Cleaned dataset saved to:", cleaned_csv_path)
print("Final cleaned shape:", df.shape)